# VirTues CPU compatibility preflight

This notebook checks whether the existing VirTues source can import through the reversible CPU SDPA fallback, verifies the pinned checkpoint file hash/size/tensor count, and attempts a strict model load plus a **tiny synthetic forward only**.

**It does not load HCC images or masks. A pass is not biological validation and does not authorize official VirTues inference on the released normalized [0,1] HCC TIFFs.** The fallback makes a timestamped source backup before changing the attention module. Run this in the Colab runtime that already contains the source, checkpoint and marker embeddings. Review output before proceeding.


In [ ]:
from pathlib import Path
import sys

VIRTUES_ROOT = Path('/content/virtues_official')
CHECKPOINT = Path('/content/hcc_virtues_private/checkpoints/virtues-imc14/model.safetensors')
MARKER_EMBEDDINGS = VIRTUES_ROOT / 'assets/example_dataset/marker_embeddings'
HELPER_REPO = Path('/content/cancer-immunotherapy-robust-discovery-audit')

for label, path in {
    'VirTues source': VIRTUES_ROOT,
    'Checkpoint': CHECKPOINT,
    'Example ESM marker embeddings': MARKER_EMBEDDINGS,
}.items():
    print(f'{label}: {path} | exists={path.exists()}')

if not VIRTUES_ROOT.is_dir() or not CHECKPOINT.is_file() or not MARKER_EMBEDDINGS.is_dir():
    raise FileNotFoundError('One or more required paths are missing. Correct the paths first; do not download duplicate or replace the checkpoint blindly.')


In [ ]:
import subprocess

REPO_URL = 'https://github.com/culprit-k/cancer-immunotherapy-robust-discovery.git'
BRANCH = 'hcc-virtues-audit-next-step'

if not HELPER_REPO.exists():
    subprocess.run(['git', 'clone', '--single-branch', '--branch', BRANCH, REPO_URL, str(HELPER_REPO)], check=True)
else:
    if not (HELPER_REPO / '.git').exists():
        raise RuntimeError(f'{HELPER_REPO} exists but is not a Git checkout. Rename it or choose another helper path.')
    dirty = subprocess.run(['git','-C',str(HELPER_REPO),'status','--porcelain'], check=True, capture_output=True, text=True).stdout.strip()
    if dirty:
        raise RuntimeError('The helper checkout has local edits. Review them before updating; no pull was run.')
    subprocess.run(['git','-C',str(HELPER_REPO),'fetch','origin',BRANCH], check=True)
    subprocess.run(['git','-C',str(HELPER_REPO),'checkout',BRANCH], check=True)
    subprocess.run(['git','-C',str(HELPER_REPO),'pull','--ff-only','origin',BRANCH], check=True)

print('Helper checkout ready:', HELPER_REPO)
print(subprocess.run(['git','-C',str(HELPER_REPO),'rev-parse','HEAD'], check=True, capture_output=True, text=True).stdout.strip())


## Run the reversible patch and synthetic preflight

The next cell applies the compatibility patch only to `virtues/modules/layers/attention_flashattention.py`, and it writes a timestamped `.bak` beside that file. It then checks checkpoint identity and attempts a strict model state-dictionary load with the repository's default config and example marker embeddings. If any gate fails, inspect the JSON report; do not attempt HCC inference.

In [ ]:
import subprocess, sys

REPORT = Path('/content/virtues_cpu_preflight_report.json')
command = [
    sys.executable,
    str(HELPER_REPO / 'scripts/run_virtues_cpu_preflight_colab.py'),
    '--virtues-root', str(VIRTUES_ROOT),
    '--checkpoint', str(CHECKPOINT),
    '--marker-embeddings', str(MARKER_EMBEDDINGS),
    '--apply-patch',
    '--out', str(REPORT),
]
result = subprocess.run(command, check=False, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print('STDERR:\n', result.stderr)
print('RETURN CODE:', result.returncode)
print('REPORT:', REPORT, '| exists=', REPORT.exists())


In [ ]:
import json

if not REPORT.is_file():
    raise FileNotFoundError(f'Preflight did not write its report: {REPORT}')
report = json.loads(REPORT.read_text())
print('OVERALL:', report.get('overall'))
print('\nSTEP STATUS')
for name, item in report.get('steps', {}).items():
    print(f"{name}: {item.get('status')}")
print('\nSCIENTIFIC INFERENCE:', report.get('scientific_inference'))
print('BIOLOGICAL CLAIMS:', report.get('biological_claims'))
print('\nReport path:', REPORT)
print('Keep this report private when it includes local paths. Send back the printed step statuses and any error traceback, not the HCC image files.')


## How to interpret the result

- `PASS_SYNTHETIC_PREFLIGHT_ONLY` means the checkpoint hash matched, strict state-dict loading passed, and a tiny synthetic input produced finite output on the available CPU runtime.
- `BLOCKED_OR_FAILED` means inspect the exact failed step and traceback. Do not ignore hash mismatches or change architecture dimensions merely to make loading succeed.
- Neither result establishes official HCC input compatibility, zero training exposure, a biological target, a causal mechanism, clinical prediction performance, novelty, or patentability.

**Do not run the model on the released normalized HCC TIFFs yet.** Separate gates still require authoritative channel-to-plane mapping, resolution of ICI_24/ICI_39 ROI multiplicity, mask-label semantics, checkpoint-exposure evidence, and a defensible preprocessing contract.